<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

In [1]:
# Parameters
variant = "feature"
ref = "70e0e18e6737736fe307c5b8b8d65f092c2e7002"
seed = 0


# Proposed validation — review before it runs

**What this measures:** Since no paper Dice value is available in context (BraTS training/eval is entirely absent from this PR and the paper's Table 2-4 rows were not furnished), the target instantiates the claim's own fallback: the relative error between the implementation's actual parameter count (built via the real `SegFormer3D` class, BraTS-style 4-channel/128³ input) and the paper's reported 4.5M — the only concrete "parity with the reference" number this PR's own VALIDATION.md supplies.

**Target metric:** `param_count_relative_error`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_segformer3d_brats_parity.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [2]:
#!/usr/bin/env python
"""
Evaluation: does the SegFormer3D implementation reach architecture-size parity
with the paper's reported BraTS-style configuration?

The maintainer conversation asks for "parity ... on BraTS", but the PR itself
(see VALIDATION.md) explicitly defers trained-Dice parity to future GPU work.
The only quantitative, non-invented parity evidence the PR supplies is its own
params/GFLOPs cross-check table (4.5M paper vs 4.25M impl at a 4-channel,
128^3 BraTS-style input). This script reproduces that architecture-size proxy
against the REAL `SegFormer3D` module (not a reimplementation), and separately
guards that adding the module did not break importing the rest of the nets zoo.
"""

'\nEvaluation: does the SegFormer3D implementation reach architecture-size parity\nwith the paper\'s reported BraTS-style configuration?\n\nThe maintainer conversation asks for "parity ... on BraTS", but the PR itself\n(see VALIDATION.md) explicitly defers trained-Dice parity to future GPU work.\nThe only quantitative, non-invented parity evidence the PR supplies is its own\nparams/GFLOPs cross-check table (4.5M paper vs 4.25M impl at a 4-channel,\n128^3 BraTS-style input). This script reproduces that architecture-size proxy\nagainst the REAL `SegFormer3D` module (not a reimplementation), and separately\nguards that adding the module did not break importing the rest of the nets zoo.\n'

In [3]:
from __future__ import annotations

import argparse
import json
import os
import sys

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
import torch

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Paper's reported parameter count for the default (paper-cited) configuration
# at a 4-channel, 128^3 BraTS-style input (Table cited in VALIDATION.md).
PAPER_PARAM_COUNT = 4.5e6

NameError: name '__file__' is not defined

In [ ]:
# Paper-default config, exactly as documented in VALIDATION.md / the PR docstring.
DEFAULT_CONFIG = dict(
    in_channels=4,
    out_channels=3,
    embed_dims=(32, 64, 160, 256),
    depths=(2, 2, 2, 2),
    num_heads=(1, 2, 5, 8),
    sr_ratios=(4, 2, 1, 1),
    mlp_ratios=(4, 4, 4, 4),
    decoder_head_embedding_dim=128,
)

In [ ]:
BRATS_INPUT_SHAPE = (1, 4, 128, 128, 128)  # held-constant BraTS-style geometry

def try_import_segformer3d():
    try:
        from monai.networks.nets import SegFormer3D  # real, changed module

        return SegFormer3D
    except Exception:
        return None

In [ ]:
def measure_param_count_and_forward(segformer3d_cls) -> tuple[int, bool]:
    """Instantiate the real module with paper defaults, count params, and run a
    deterministic forward smoke check at the held-constant BraTS geometry."""
    if segformer3d_cls is None:
        return 0, False
    try:
        torch.manual_seed(0)
        net = segformer3d_cls(**DEFAULT_CONFIG)
        param_count = int(sum(p.numel() for p in net.parameters()))

        net.eval()
        torch.manual_seed(0)
        x = torch.randn(*BRATS_INPUT_SHAPE)
        with torch.no_grad():
            y = net(x)
        expected_shape = (BRATS_INPUT_SHAPE[0], DEFAULT_CONFIG["out_channels"]) + BRATS_INPUT_SHAPE[2:]
        forward_ok = tuple(y.shape) == expected_shape
        return param_count, forward_ok
    except Exception:
        return 0, False

In [ ]:
def measure_existing_nets_import_success_rate() -> float:
    """Guardrail: adding segformer3d.py (and its `nets/__init__.py` import line)
    must not break importing the pre-existing nets that share that same module."""
    existing_nets = [
        "UNet",
        "BasicUNet",
        "SegResNet",
        "DynUNet",
        "AttentionUnet",
        "VNet",
        "UNETR",
        "SwinUNETR",
        "VarAutoEncoder",
    ]
    successes = 0
    for name in existing_nets:
        try:
            mod = __import__("monai.networks.nets", fromlist=[name])
            getattr(mod, name)
            successes += 1
        except Exception:
            pass
    return successes / len(existing_nets)

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", default=None)
    parser.parse_args()

    torch.manual_seed(0)

    segformer3d_cls = try_import_segformer3d()
    param_count, forward_ok = measure_param_count_and_forward(segformer3d_cls)

    # Relative error vs. the paper's published parameter count for this exact
    # config/geometry. On baseline (module absent) param_count=0 -> error=1.0,
    # a maximally degraded (but still finite, non-crashing) value.
    param_count_relative_error = abs(PAPER_PARAM_COUNT - param_count) / PAPER_PARAM_COUNT

    existing_nets_import_success_rate = measure_existing_nets_import_success_rate()

    metrics = {
        "param_count_relative_error": round(param_count_relative_error, 6),
        "existing_nets_import_success_rate": round(existing_nets_import_success_rate, 6),
        "param_count": param_count,
        "forward_shape_ok": forward_ok,
    }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: segformer3d-brats-architecture-parity
    suite: "eval/eval_segformer3d_brats_parity.py"
    scorer: param_count_relative_error
    baseline: main
    metrics:
      - name: param_count_relative_error
        role: target
        direction: min
        threshold: 0.056
      - name: existing_nets_import_success_rate
        role: guardrail
        direction: max
        threshold: 1.0
    policy: {guardrail_veto: true}
held_constant:
  - "input geometry: (1, 4, 128, 128, 128), the paper's BraTS-style 4-channel volume at 128 cubed"
  - "default paper-cited hyperparameters: embed_dims=(32,64,160,256), depths=(2,2,2,2), num_heads=(1,2,5,8), sr_ratios=(4,2,1,1), mlp_ratios=(4,4,4,4), decoder_head_embedding_dim=128"
  - "paper reference parameter figure of 4.5M used as the fixed comparison constant on both arms"
  - "torch seed fixed for the forward-pass smoke check"
avoid:
  - "no BraTS dataset download, loss, or optimizer step is exercised: the PR itself defers full Dice-parity training to future GPU work, so this script measures the architecture-size proxy the PR's own VALIDATION.md reports, not trained accuracy"
  - "no wall-clock timing is used as a metric; only parameter counts and a shape check are measured"
  - "no invented Dice numbers: BraTS mean Dice is left out of the metrics entirely because no paper table row is present in context"
compute:
  tier: cpu
provenance:
  param_count_relative_error: "user_guidance (BraTS parity ask) refined by claim_analysis's architecture-proxy fallback, numeric anchor from PR VALIDATION.md cross-check table (4.5M paper vs 4.25M impl)"
  existing_nets_import_success_rate: "inferred — regression guardrail for the `monai/networks/nets/__init__.py` edit this PR makes, since that file is shared by every existing net"
  held_constant: "protocol_doc:monai/networks/nets/segformer3d.py (VALIDATION.md paper-default config table)"
  suite: "synthesized (R1 maturity repo: tests + CI only, no BraTS benchmark harness exists to run (a) against)"
```